# Executive Inventory & Supply Chain Analytics Dashboard

## Data Preparation & Feature Engineering

### Author
Sachin Kumar Gupta

### Tools & Technologies
- Python
- Pandas
- NumPy
- Google Colab
- Tableau
- SQL

---

## Project Overview

This project simulates an end-to-end analytics solution for a medium sized retail company that manages thousands of products across multiple warehouses and suppliers.

The organization faces common supply chain challenges, including stock shortages, excess inventory, delayed supplier deliveries, and inconsistent warehouse utilization. These operational inefficiencies impact customer satisfaction, increase inventory carrying costs, and reduce profitability.

This notebook focuses on preparing raw ERP style data into a clean, validated, and analytics ready data model that will be used for SQL analysis and interactive Tableau dashboards.

---

## Business Objectives

The analytics solution aims to help management answer the following business questions:

### Inventory Management
- Which products are at risk of stockout?
- Which products are overstocked?
- Which product categories contribute the most inventory value?
- What is the current inventory health across warehouses?

### Procurement & Supplier Performance
- Which suppliers consistently deliver late?
- Which suppliers have the highest defect rates?
- How do suppliers compare based on lead time and reliability?

### Warehouse Operations
- Which warehouse has the highest inventory utilization?
- Which warehouse holds excess inventory?
- How efficiently is inventory distributed across locations?

### Sales & Demand Planning
- Which products generate the highest revenue and profit?
- How accurate are demand forecasts?
- What seasonal patterns influence customer demand?

---

## Project Workflow

1. Load Raw ERP Data
2. Perform Data Quality Assessment
3. Clean and Standardize Data
4. Validate Relationships and Business Rules
5. Engineer Business Metrics
6. Build Analytics Data Model
7. Export Tableau-Ready Datasets

# Dataset Overview

The project uses eight ERP-style datasets that represent different business functions within the supply chain.

| Dataset | Description | Granularity |
|----------|-------------|-------------|
| Product Master | Product catalog and inventory configuration | One row per product |
| Supplier Master | Supplier information and performance metrics | One row per supplier |
| Warehouse Master | Warehouse details and storage capacity | One row per warehouse |
| Purchase Orders | Procurement transactions | One row per purchase order |
| Inventory Transactions | Inventory movement history | One row per inventory transaction |
| Sales Orders | Customer sales transactions | One row per sales order |
| Inventory Snapshot | Daily inventory levels | Product × Warehouse × Snapshot Date |
| Demand Forecast | Forecasted and actual demand | Product × Forecast Date |

# Import Required Libraries

In [ ]:
import os
from pathlib import Path
import numpy as np
import pandas as pd

# Display Settings
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.float_format", "{:.2f}".format)

import warnings
warnings.filterwarnings("ignore")

print("Libraries imported successfully.")

Libraries imported successfully.


In [ ]:
# ============================================================
# Project Configuration
# ============================================================

PROJECT_NAME = "Executive Inventory & Supply Chain Analytics Dashboard"

RAW_DATA_PATH = "/content/"
OUTPUT_PATH = "/content/processed_data/"

# Create output directory if it doesn't exist
os.makedirs(OUTPUT_PATH, exist_ok=True)

print(f"Project : {PROJECT_NAME}")
print(f"Raw Data Folder : {RAW_DATA_PATH}")
print(f"Output Folder : {OUTPUT_PATH}")

Project : Executive Inventory & Supply Chain Analytics Dashboard
Raw Data Folder : /content/
Output Folder : /content/processed_data/


# Data Loading


This section will loads all raw ERP datasets into pandas DataFrames.

Each dataset represents a separate business entity and will later be validated using referential integrity checks before feature engineering begins.

In [ ]:
DATASETS = {
    "product": "product_master.csv",
    "supplier": "supplier_master.csv",
    "warehouse": "warehouse_master.csv",
    "purchase": "purchase_orders.csv",
    "transactions": "inventory_transactions.csv",
    "sales": "sales_orders.csv",
    "snapshot": "daily_inventory_snapshot.csv",
    "forecast": "demand_forecast.csv"
}

# ============================================================
# Function to Load CSV Files
# ============================================================

def load_dataset(folder_path, file_name):
    """
    Load a CSV dataset from the specified folder.

    Parameters
    ----------
    folder_path : str
        Folder containing the dataset

    file_name : str
        CSV file name

    Returns
    -------
    pandas.DataFrame
    """

    file_path = os.path.join(folder_path, file_name)

    if not os.path.exists(file_path):
        raise FileNotFoundError(f"{file_name} not found.")

    df = pd.read_csv(file_path)

    print(f"Loaded {file_name:<30} Shape: {df.shape}")

    return df

In [ ]:
# ============================================================
# Load All ERP Datasets
# ============================================================

product = load_dataset(RAW_DATA_PATH, DATASETS["product"])
supplier = load_dataset(RAW_DATA_PATH, DATASETS["supplier"])
warehouse = load_dataset(RAW_DATA_PATH, DATASETS["warehouse"])
purchase = load_dataset(RAW_DATA_PATH, DATASETS["purchase"])
transactions = load_dataset(RAW_DATA_PATH, DATASETS["transactions"])
sales = load_dataset(RAW_DATA_PATH, DATASETS["sales"])
snapshot = load_dataset(RAW_DATA_PATH, DATASETS["snapshot"])
forecast = load_dataset(RAW_DATA_PATH, DATASETS["forecast"])

Loaded product_master.csv             Shape: (5000, 25)
Loaded supplier_master.csv            Shape: (200, 18)
Loaded warehouse_master.csv           Shape: (5, 12)
Loaded purchase_orders.csv            Shape: (80410, 14)
Loaded inventory_transactions.csv     Shape: (374576, 9)
Loaded sales_orders.csv               Shape: (295838, 16)
Loaded daily_inventory_snapshot.csv   Shape: (2575000, 12)
Loaded demand_forecast.csv            Shape: (26250, 8)


In [ ]:
# ============================================================
# Store All DataFrames in a Dictionary
# ============================================================

datasets = {
    "Product Master": product,
    "Supplier Master": supplier,
    "Warehouse Master": warehouse,
    "Purchase Orders": purchase,
    "Inventory Transactions": transactions,
    "Sales Orders": sales,
    "Inventory Snapshot": snapshot,
    "Demand Forecast": forecast
}

In [ ]:
summary = []

for name, df in datasets.items():

    summary.append({
        "Dataset": name,
        "Rows": df.shape[0],
        "Columns": df.shape[1],
        "Missing Values": int(df.isna().sum().sum()),
        "Duplicate Rows": int(df.duplicated().sum())
    })

summary_df = pd.DataFrame(summary)

summary_df

,Dataset,Rows,Columns,Missing Values,Duplicate Rows
0,Product Master,5000,25,0,0
1,Supplier Master,200,18,0,0
2,Warehouse Master,5,12,0,0
3,Purchase Orders,80410,14,4153,0
4,Inventory Transactions,374576,9,0,0
5,Sales Orders,295838,16,0,2929
6,Inventory Snapshot,2575000,12,0,0
7,Demand Forecast,26250,8,0,0


In [ ]:
# ============================================================
# Overall Dataset Statistics
# ============================================================

total_rows = sum(df.shape[0] for df in datasets.values())
total_columns = sum(df.shape[1] for df in datasets.values())

print("=" * 60)
print("PROJECT DATA SUMMARY")
print("=" * 60)
print(f"Total Datasets : {len(datasets)}")
print(f"Total Records  : {total_rows:,}")
print(f"Total Columns  : {total_columns}")

PROJECT DATA SUMMARY
Total Datasets : 8
Total Records  : 3,357,279
Total Columns  : 114


# Data Profiling

Before cleaning the data, it is important to understand the structure and quality of each dataset.

This section provides a high-level profile of every table, including:

- Number of rows and columns
- Memory usage
- Data types
- Missing values
- Duplicate records
- Unique values
- Numeric statistics
- Categorical statistics

The goal is to identify potential data quality issues before performing transformations.

In [ ]:
# ============================================================
# Dataset Profile Function
# ============================================================

def dataset_profile(df, dataset_name):
    """
    Display a high-level profile of a dataset.
    """

    print("=" * 80)
    print(dataset_name.upper())
    print("=" * 80)

    rows, cols = df.shape

    print(f"Rows                : {rows:,}")
    print(f"Columns             : {cols}")

    memory = df.memory_usage(deep=True).sum() / 1024**2
    print(f"Memory Usage        : {memory:.2f} MB")

    print(f"Duplicate Rows      : {df.duplicated().sum():,}")

    print(f"Missing Values      : {df.isna().sum().sum():,}")

    print("\nData Types")

    print(df.dtypes.value_counts())

    print("\nFirst Five Rows")

    display(df.head())

In [ ]:
for name, df in datasets.items():
    dataset_profile(df, name)

PRODUCT MASTER
Rows                : 5,000
Columns             : 25
Memory Usage        : 4.23 MB
Duplicate Rows      : 0
Missing Values      : 0

Data Types
object     14
int64       6
float64     5
Name: count, dtype: int64

First Five Rows


,Product_ID,SKU,Barcode,Product_Name,Category,Subcategory,Brand,Unit_Cost,Selling_Price,Gross_Margin_Pct,Weight,Volume,Shelf_Life_Days,ABC_Class,Preferred_Supplier_ID,Default_Warehouse,Launch_Date,Discontinued_Flag,Reorder_Level,Maximum_Stock,Safety_Stock,Economic_Order_Quantity,Storage_Type,Product_Status,Active_Inactive
0,PROD00001,SPO-GOL-00001,789769991378,Golf Model B1,Sports,Golf,Brand_23,18.21,26.34,30.87,1.84,1.74,365,B,SUP0100,WH003,2020-08-23,No,98,294,40,208,Ambient,Active,Active
1,PROD00002,FAS-FOO-00002,789332968651,Footwear Model C2,Fashion,Footwear,Brand_22,14.55,21.65,32.79,0.31,0.09,730,B,SUP0059,WH002,2021-02-25,No,87,174,47,219,Ambient,Active,Active
2,PROD00003,HOM-BED-00003,789874414982,Bedding Model D3,Home & Kitchen,Bedding,Brand_21,9.12,13.65,33.19,13.57,0.08,365,C,SUP0089,WH004,2022-06-02,No,63,126,7,236,Ambient,Active,Active
3,PROD00004,FUR-TAB-00004,789887110843,Tables Model E4,Furniture,Tables,Brand_7,37.97,53.90,29.55,54.65,5.25,9999,B,SUP0035,WH001,2020-12-01,No,86,172,39,135,Ambient,Active,Active
4,PROD00005,OFF-ORG-00005,789496360081,Organization Model F5,Office Supplies,Organization,Brand_26,7.99,11.21,28.72,13.55,0.82,730,C,SUP0014,WH004,2020-07-14,No,81,162,8,285,Ambient,Active,Active


SUPPLIER MASTER
Rows                : 200
Columns             : 18
Memory Usage        : 0.11 MB
Duplicate Rows      : 0
Missing Values      : 0

Data Types
object     8
float64    6
int64      4
Name: count, dtype: int64

First Five Rows


,Supplier_ID,Supplier_Name,Country,Region,Years_with_Company,Supplier_Rating,Lead_Time_Days,Minimum_Order_Qty,Average_Order_Value,Contract_Start_Date,Contract_End_Date,On_Time_Delivery_Pct,Quality_Rating,Defect_Rate_Pct,Late_Delivery_Pct,Cost_Rating,Preferred_Supplier,Contact_Email
0,SUP0001,Global Sourcing Partner 1,Canada,North America,11,4.56,9,100,83357.32,2020-11-26,2026-01-29,97.86,4.38,1.43,2.14,4,Yes,fulfillment@globalsourcing1.com
1,SUP0002,Global Sourcing Partner 2,Germany,EMEA,1,4.72,6,250,61213.82,2018-11-08,2026-04-01,92.16,4.53,1.09,7.84,5,Yes,fulfillment@globalsourcing2.com
2,SUP0003,Global Sourcing Partner 3,Germany,EMEA,3,4.30,14,250,52738.40,2019-07-06,2027-08-20,96.20,4.13,0.13,3.80,5,Yes,fulfillment@globalsourcing3.com
3,SUP0004,Global Sourcing Partner 4,Vietnam,APAC,9,4.71,9,250,57006.07,2018-08-18,2026-07-21,96.86,4.52,0.17,3.14,4,Yes,fulfillment@globalsourcing4.com
4,SUP0005,Global Sourcing Partner 5,Japan,APAC,1,4.57,11,250,23854.54,2020-01-22,2025-08-02,93.85,4.39,0.12,6.15,5,Yes,fulfillment@globalsourcing5.com


WAREHOUSE MASTER
Rows                : 5
Columns             : 12
Memory Usage        : 0.00 MB
Duplicate Rows      : 0
Missing Values      : 0

Data Types
object     6
int64      3
float64    3
Name: count, dtype: int64

First Five Rows


...

,Order_ID,Order_Date,Customer_ID,Product_ID,Warehouse_ID,Quantity,Selling_Price,Discount_Pct,Revenue,COGS,Profit,Order_Status,Shipping_Method,Delivery_Time_Days,Returned,Customer_State
0,SO00000001,2023-01-01,CUST36646,PROD01861,WH002,2,99.15,0.00,198.30,141.74,56.56,Completed,Expedited,3,No,FL
1,SO00000002,2023-01-01,CUST19823,PROD04768,WH001,1,93.51,0.00,93.51,63.82,29.69,Completed,Expedited,3,No,FL
2,SO00000003,2023-01-01,CUST98223,PROD03682,WH003,2,85.79,0.00,171.58,117.16,54.42,Completed,Two-Day Air,6,No,MO
3,SO00000004,2023-01-01,CUST48304,PROD03017,WH002,1,44.02,0.00,44.02,32.66,11.36,Completed,Standard Ground,3,No,WA
4,SO00000005,2023-01-01,CUST37350,PROD00755,WH001,4,1.70,0.00,0.00,4.80,-4.80,Cancelled,Expedited,3,No,MI


INVENTORY SNAPSHOT
Rows                : 2,575,000
Columns             : 12
Memory Usage        : 719.89 MB
Duplicate Rows      : 0
Missing Values      : 0

Data Types
int64      5
object     4
float64    3
Name: count, dtype: int64

First Five Rows


,Snapshot_Date,Warehouse_ID,Product_ID,Opening_Stock,Received_Qty,Sold_Qty,Adjustment_Qty,Closing_Stock,Inventory_Value,Days_of_Inventory,Inventory_Status,Capacity_Utilization_Pct
0,2023-01-08,WH001,PROD00001,44,84,13,-2,113,2057.73,60.80,Healthy,645.64
1,2023-01-08,WH001,PROD00002,160,38,10,0,188,2735.40,131.60,Overstock,645.64
2,2023-01-08,WH001,PROD00003,82,15,3,0,94,857.28,188.00,Healthy,645.64
3,2023-01-08,WH001,PROD00004,85,38,8,-1,114,4328.58,99.80,Healthy,645.64
4,2023-01-08,WH001,PROD00005,69,19,3,0,85,679.15,170.00,Healthy,645.64


DEMAND FORECAST
Rows                : 26,250
Columns             : 8
Memory Usage        : 7.40 MB
Duplicate Rows      : 0
Missing Values      : 0

Data Types
object     5
int64      2
float64    1
Name: count, dtype: int64

First Five Rows


,Forecast_Date,Product_ID,Forecast_Qty,Actual_Qty,Forecast_Error_Pct,Seasonality,Promotion,Holiday
0,2023-01-01,PROD01502,6,7,14.29,Low,No,No
1,2023-01-01,PROD02587,106,99,7.07,Low,No,No
2,2023-01-01,PROD02654,4,5,20.00,Low,No,No
3,2023-01-01,PROD01056,21,22,4.55,Low,No,No
4,2023-01-01,PROD00706,22,22,0.00,Low,No,No


In [ ]:
# ============================================================
# Data Quality Summary
# ============================================================

quality_summary = []

for name, df in datasets.items():

    quality_summary.append({

        "Dataset": name,

        "Rows": df.shape[0],

        "Columns": df.shape[1],

        "Missing Values": df.isna().sum().sum(),

        "Duplicate Rows": df.duplicated().sum(),

        "Memory (MB)": round(df.memory_usage(deep=True).sum()/1024**2,2)

    })

quality_summary = pd.DataFrame(quality_summary)

quality_summary

,Dataset,Rows,Columns,Missing Values,Duplicate Rows,Memory (MB)
0,Product Master,5000,25,0,0,4.23
1,Supplier Master,200,18,0,0,0.11
2,Warehouse Master,5,12,0,0,0.00
3,Purchase Orders,80410,14,4153,0,42.46
4,Inventory Transactions,374576,9,0,0,153.22
5,Sales Orders,295838,16,0,2929,159.67
6,Inventory Snapshot,2575000,12,0,0,719.89
7,Demand Forecast,26250,8,0,0,7.40


In [ ]:
# ============================================================
# Numeric Columns Summary
# ============================================================

for name, df in datasets.items():

    numeric = df.select_dtypes(include=np.number)

    if not numeric.empty:

        print("="*80)

        print(name)

        display(numeric.describe().T)

Product Master


,count,mean,std,min,25%,50%,75%,max
Barcode,5000.00,789544509711.57,259175122.44,789100131936.00,789323115235.50,789537172370.00,789767110494.25,789999781633.00
Unit_Cost,5000.00,52.20,125.44,1.03,7.50,15.26,51.19,1170.96
Selling_Price,5000.00,69.82,158.73,1.44,10.63,21.37,72.66,1471.13
Gross_Margin_Pct,5000.00,28.09,4.96,13.06,25.38,28.97,31.76,37.49
Weight,5000.00,19.22,15.02,0.10,8.61,17.26,25.53,79.97
Volume,5000.00,1.37,1.15,0.05,0.61,1.16,1.70,5.99
Shelf_Life_Days,5000.00,4874.95,4777.84,7.00,365.00,730.00,9999.00,9999.00
Reorder_Level,5000.00,90.61,50.60,25.00,57.00,78.00,102.00,277.00
Maximum_Stock,5000.00,225.61,135.72,50.00,135.00,188.00,267.00,831.00
Safety_Stock,5000.00,41.21,47.84,5.00,11.00,19.00,45.00,199.00


Supplier Master


,count,mean,std,min,25%,50%,75%,max
Years_with_Company,200.00,5.66,2.98,1.00,3.00,5.00,8.00,11.00
Supplier_Rating,200.00,3.56,0.63,2.52,3.09,3.55,3.98,4.97
Lead_Time_Days,200.00,27.04,11.40,5.00,18.00,27.00,38.00,44.00
Minimum_Order_Qty,200.00,186.60,180.44,10.00,50.00,100.00,250.00,500.00
Average_Order_Value,200.00,51069.21,24991.78,8657.97,29489.48,52728.02,71752.96,94749.83
On_Time_Delivery_Pct,200.00,82.22,8.26,70.13,75.26,81.14,88.67,99.49
Quality_Rating,200.00,3.42,0.60,2.42,2.97,3.41,3.82,4.77
Defect_Rate_Pct,200.00,3.82,2.01,0.12,2.16,3.97,5.27,7.48
Late_Delivery_Pct,200.00,17.78,8.26,0.51,11.33,18.86,24.74,29.87
Cost_Rating,200.00,3.15,1.00,2.00,2.00,3.00,4.00,5.00


Warehouse Master


...

,count,mean,std,min,25%,50%,75%,max
Quantity,295838.00,1.55,0.86,1.00,1.00,1.00,2.00,5.00
Selling_Price,295838.00,118.37,228.51,1.44,13.20,39.64,106.99,1471.13
Discount_Pct,295838.00,2.49,5.62,0.00,0.00,0.00,0.00,25.00
Revenue,295838.00,128.31,220.72,0.00,19.62,48.29,123.98,1471.13
COGS,295838.00,100.25,180.32,1.03,15.18,36.14,94.58,1170.96
Profit,295838.00,28.06,57.88,-1170.96,4.86,12.30,31.64,328.56
Delivery_Time_Days,295838.00,3.90,1.58,1.00,3.00,4.00,5.00,6.00


Inventory Snapshot


,count,mean,std,min,25%,50%,75%,max
Opening_Stock,2575000.00,115.51,106.76,0.00,45.00,91.00,155.00,1045.00
Received_Qty,2575000.00,38.73,35.56,0.00,14.00,30.00,52.00,289.00
Sold_Qty,2575000.00,11.53,10.54,0.00,4.00,6.00,15.00,69.00
Adjustment_Qty,2575000.00,-0.15,0.57,-2.00,0.00,0.00,0.00,1.00
Closing_Stock,2575000.00,140.58,116.36,0.00,64.00,114.00,183.00,1033.00
Inventory_Value,2575000.00,11103.74,40331.15,0.00,578.88,1512.80,6026.11,1045148.52
Days_of_Inventory,2575000.00,118.78,96.91,0.00,49.00,93.50,163.60,1718.00
Capacity_Utilization_Pct,2575000.00,773.09,112.98,623.03,688.53,746.74,816.31,998.16


Demand Forecast


,count,mean,std,min,25%,50%,75%,max
Forecast_Qty,26250.00,35.33,41.57,1.00,7.00,19.00,45.00,277.00
Actual_Qty,26250.00,35.74,40.57,1.00,8.00,19.00,45.00,208.00
Forecast_Error_Pct,26250.00,12.71,11.60,0.00,3.06,11.11,19.23,71.43


In [ ]:
# ============================================================
# Categorical Columns Summary
# ============================================================

for name, df in datasets.items():

    categorical = df.select_dtypes(include="object")

    if len(categorical.columns) == 0:
        continue

    print("=" * 80)
    print(name)

    summary = pd.DataFrame({

        "Unique Values": categorical.nunique(),

        "Most Frequent": categorical.mode().iloc[0],

        "Frequency": categorical.apply(lambda x: x.value_counts().iloc[0])

    })

    display(summary)

Product Master


,Unique Values,Most Frequent,Frequency
Product_ID,5000,PROD00001,1
SKU,5000,BEA-COS-00020,1
Product_Name,5000,Accessories Model A1170,1
Category,8,Furniture,650
Subcategory,39,Accessories,266
Brand,50,Brand_31,121
ABC_Class,3,C,2500
Preferred_Supplier_ID,200,SUP0149,38
Default_Warehouse,5,WH005,1055
Launch_Date,798,2021-01-08,19


Supplier Master


,Unique Values,Most Frequent,Frequency
Supplier_ID,200,SUP0001,1
Supplier_Name,200,Global Sourcing Partner 1,1
Country,8,Vietnam,34
Region,5,APAC,88
Contract_Start_Date,192,2017-03-14,2
Contract_End_Date,176,2027-05-19,3
Preferred_Supplier,2,No,170
Contact_Email,200,fulfillment@globalsourcing1.com,1


Warehouse Master


...

,Unique Values,Most Frequent,Frequency
Order_ID,292909,SO00000031,2
Order_Date,731,2023-12-08,628
Customer_ID,86545,CUST20862,14
Product_ID,5000,PROD04027,169
Warehouse_ID,5,WH004,59357
Order_Status,2,Completed,289793
Shipping_Method,4,Standard Ground,148210
Returned,2,No,288703
Customer_State,20,GA,15055


Inventory Snapshot


,Unique Values,Most Frequent,Frequency
Snapshot_Date,103,2023-01-08,25000
Warehouse_ID,5,WH001,515000
Product_ID,5000,PROD00001,515
Inventory_Status,4,Healthy,1673395


Demand Forecast


,Unique Values,Most Frequent,Frequency
Forecast_Date,105,2023-01-01,250
Product_ID,250,PROD00009,105
Seasonality,3,Low,13250
Promotion,2,No,20000
Holiday,2,No,21750


In [ ]:
# ============================================================
# Potential Date Columns
# ============================================================

print("Potential Date Columns\n")

for name, df in datasets.items():

    date_cols = [col for col in df.columns if "date" in col.lower()]

    if date_cols:

        print(f"{name}")

        print(date_cols)

        print("-"*40)

Potential Date Columns

Product Master
['Launch_Date']
----------------------------------------
Supplier Master
['Contract_Start_Date', 'Contract_End_Date']
----------------------------------------
Purchase Orders
['PO_Date', 'Expected_Delivery_Date', 'Actual_Delivery_Date']
----------------------------------------
Inventory Transactions
['Transaction_Date']
----------------------------------------
Sales Orders
['Order_Date']
----------------------------------------
Inventory Snapshot
['Snapshot_Date']
----------------------------------------
Demand Forecast
['Forecast_Date']
----------------------------------------


In [ ]:
for name, df in datasets.items():

    duplicates = df.duplicated().sum()

    print(f"{name}: {duplicates} duplicate rows")

Product Master: 0 duplicate rows
Supplier Master: 0 duplicate rows
Warehouse Master: 0 duplicate rows
Purchase Orders: 0 duplicate rows
Inventory Transactions: 0 duplicate rows
Sales Orders: 2929 duplicate rows
Inventory Snapshot: 0 duplicate rows
Demand Forecast: 0 duplicate rows


In [ ]:
purchase_orders = datasets["Purchase Orders"]

purchase_orders.isnull().sum().sort_values(ascending=False)

,0
Actual_Delivery_Date,4153
PO_ID,0
Supplier_ID,0
Warehouse_ID,0
Product_ID,0
PO_Date,0
Ordered_Qty,0
Received_Qty,0
Unit_Cost,0
Expected_Delivery_Date,0


In [ ]:
purchase_orders[
    purchase_orders["Actual_Delivery_Date"].isna()
]["Purchase_Status"].value_counts()

,count
Purchase_Status,
Pending,2953
Cancelled,1200


In [ ]:
sales_orders = datasets["Sales Orders"]

sales_orders.duplicated().sum()

np.int64(2929)

In [ ]:
sales_orders[
    sales_orders.duplicated(keep=False)
].sort_values("Order_ID")

,Order_ID,Order_Date,Customer_ID,Product_ID,Warehouse_ID,Quantity,Selling_Price,Discount_Pct,Revenue,COGS,Profit,Order_Status,Shipping_Method,Delivery_Time_Days,Returned,Customer_State
30,SO00000031,2023-01-01,CUST16941,PROD03059,WH003,3,4.93,10.00,13.31,10.68,2.63,Completed,Standard Ground,2,No,WI
293825,SO00000031,2023-01-01,CUST16941,PROD03059,WH003,3,4.93,10.00,13.31,10.68,2.63,Completed,Standard Ground,2,No,WI
295756,SO00000089,2023-01-01,CUST10060,PROD04455,WH003,1,78.71,0.00,78.71,60.24,18.47,Completed,Expedited,3,No,OH
88,SO00000089,2023-01-01,CUST10060,PROD04455,WH003,1,78.71,0.00,78.71,60.24,18.47,Completed,Expedited,3,No,OH
172,SO00000173,2023-01-01,CUST86329,PROD02560,WH005,1,36.48,10.00,32.83,30.69,2.14,Completed,Standard Ground,2,No,IL
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
294163,SO00292694,2024-12-31,CUST20666,PROD04848,WH004,2,52.82,0.00,105.64,77.34,28.30,Completed,Standard Ground,5,No,GA
293139,SO00292741,2024-12-31,CUST42465,PROD00326,WH001,2,143.45,0.00,286.90,196.06,90.84,Completed,Two-Day Air,4,No,TX
292740,SO00292741,2024-12-31,CUST42465,PROD00326,WH001,2,143.45,0.00,286.90,196.06,90.84,Completed,Two-Day Air,4,No,TX
294558,SO00292840,2024-12-31,CUST75805,PROD00240,WH004,1,18.54,0.00,18.54,13.19,5.35,Completed,Standard Ground,4,No,VA


# Initial Observations

Based on the profiling results:

- Most datasets contain no missing values.
- Purchase Orders contains missing values in `Actual_Delivery_Date`, which are expected for pending or cancelled orders.
- Sales Orders contains duplicate records that require further investigation.
- Date columns are currently stored as text and will be converted to datetime format.
- No immediate structural issues were identified.

The next step is to perform data cleaning and validation.

# Data Cleaning & Standardization.

Raw ERP data often contains inconsistencies that can affect analysis.

Common issues include:

- Dates stored as text
- Extra spaces in text fields
- Inconsistent categorical values
- Missing operational dates
- Duplicate transactional records
- Invalid numeric values

This section performs controlled cleaning operations while preserving the original business meaning of the data.

Cleaning steps:

1. Standardize column formats
2. Convert date fields
3. Clean text columns
4. Handle missing values
5. Validate duplicate records

In [ ]:
# ============================================================
# Create Raw Data Backups
# ============================================================

product_raw = product.copy()
supplier_raw = supplier.copy()
warehouse_raw = warehouse.copy()
purchase_raw = purchase.copy()
transactions_raw = transactions.copy()
sales_raw = sales.copy()
snapshot_raw = snapshot.copy()
forecast_raw = forecast.copy()

print("Raw dataset backups created.")

Raw dataset backups created.


## Column Name Standardization

Column names are standardized using:

- Lowercase naming
- Removing spaces
- Replacing spaces with underscores

This improves consistency when using SQL and Python together.

In [ ]:
# ============================================================
# Standardize Column Names
# ============================================================

def clean_column_names(df):

    df.columns = (
        df.columns.str.strip().str.lower().str.replace(" ", "_").str.replace("-", "_")
    )

    return df


for name, df in datasets.items():

    datasets[name] = clean_column_names(df)

print("Column names standardized.")

Column names standardized.


Since we standardized columns name we have to update our dataframe variables.

In [ ]:
product = datasets["Product Master"]
supplier = datasets["Supplier Master"]
warehouse = datasets["Warehouse Master"]
purchase = datasets["Purchase Orders"]
transactions = datasets["Inventory Transactions"]
sales = datasets["Sales Orders"]
snapshot = datasets["Inventory Snapshot"]
forecast = datasets["Demand Forecast"]

print("References updated.")

References updated.


## Date Conversion

Dates are stored as strings in raw CSV exports.

Converting them into datetime format enables:

- Time-series analysis
- Monthly trends
- Year-over-year comparison
- Tableau date hierarchies

In [ ]:
# ============================================================
# Convert Date Columns
# ============================================================

date_mapping = {

    "product": [
        "launch_date"
    ],

    "supplier": [
        "contract_start_date",
        "contract_end_date"
    ],

    "purchase": [
        "po_date",
        "expected_delivery_date",
        "actual_delivery_date"
    ],

    "transactions": [
        "transaction_date"
    ],

    "sales": [
        "order_date"
    ],

    "snapshot": [
        "snapshot_date"
    ],

    "forecast": [
        "forecast_date"
    ]
}


dataframes = {

    "product": product,
    "supplier": supplier,
    "purchase": purchase,
    "transactions": transactions,
    "sales": sales,
    "snapshot": snapshot,
    "forecast": forecast
}


for name, columns in date_mapping.items():

    df = dataframes[name]

    for col in columns:

        if col in df.columns:

            df[col] = pd.to_datetime(
                df[col],
                errors="coerce"
            )


print("Date conversion completed.")

Date conversion completed.


In [ ]:
# ============================================================
# Check Date Data Types
# ============================================================

for name, df in dataframes.items():

    dates = df.select_dtypes(
        include="datetime"
    ).columns.tolist()

    if dates:

        print(name)
        print(dates)
        print()

product
['launch_date']

supplier
['contract_start_date', 'contract_end_date']

purchase
['po_date', 'expected_delivery_date', 'actual_delivery_date']

transactions
['transaction_date']

sales
['order_date']

snapshot
['snapshot_date']

forecast
['forecast_date']



## Text Standardization

Text columns may contain:

- Leading/trailing spaces
- Empty strings
- Inconsistent formatting

These issues can create duplicate categories during analysis.

In [ ]:
# ============================================================
# Clean Text Columns
# ============================================================

def clean_text_columns(df):

    text_cols = df.select_dtypes(
        include="object"
    ).columns

    for col in text_cols:

        df[col] = (
            df[col]
            .astype(str)
            .str.strip()
        )

    return df


for name, df in datasets.items():

    datasets[name] = clean_text_columns(df)


print("Text columns cleaned.")

Text columns cleaned.


## Missing Value Treatment

Missing values must be handled based on business context.

Example:

`Actual_Delivery_Date` missing in Purchase Orders does not necessarily indicate an error.

It may represent:

- Open purchase orders
- Pending deliveries
- Orders currently in transit

Therefore, missing delivery dates will be preserved.

In [ ]:
# ============================================================
# Missing Value Review
# ============================================================

for name, df in datasets.items():

    missing = df.isnull().sum()

    missing = missing[missing > 0]

    if len(missing):

        print("="*60)
        print(name)
        print(missing)

Purchase Orders
actual_delivery_date    4153
dtype: int64


In [ ]:
purchase["delivery_completed_flag"] = np.where(
    purchase["actual_delivery_date"].isna(),
    0,
    1
)

In [ ]:
# ============================================================
# Sales Duplicate Investigation
# ============================================================

sales_duplicates = sales[
    sales.duplicated(
        keep=False
    )
]

print(
    "Duplicate Sales Rows:",
    len(sales_duplicates)
)

sales_duplicates.head()

Duplicate Sales Rows: 5858


,order_id,order_date,customer_id,product_id,warehouse_id,quantity,selling_price,discount_pct,revenue,cogs,profit,order_status,shipping_method,delivery_time_days,returned,customer_state
30,SO00000031,2023-01-01,CUST16941,PROD03059,WH003,3,4.93,10.00,13.31,10.68,2.63,Completed,Standard Ground,2,No,WI
88,SO00000089,2023-01-01,CUST10060,PROD04455,WH003,1,78.71,0.00,78.71,60.24,18.47,Completed,Expedited,3,No,OH
172,SO00000173,2023-01-01,CUST86329,PROD02560,WH005,1,36.48,10.00,32.83,30.69,2.14,Completed,Standard Ground,2,No,IL
238,SO00000239,2023-01-01,CUST40933,PROD03249,WH005,1,10.06,0.00,10.06,7.42,2.64,Completed,Expedited,6,No,AZ
254,SO00000255,2023-01-01,CUST39516,PROD00816,WH002,4,2.55,5.00,9.69,7.20,2.49,Completed,Expedited,2,No,TX


In [ ]:
# ============================================================
# Exact Duplicate Check
# ============================================================

exact_duplicates = sales[
    sales.duplicated()
]

print(
    "Exact duplicate rows:",
    len(exact_duplicates)
)

Exact duplicate rows: 2929


In [ ]:
# ============================================================
# Remove Exact Duplicate Sales Transactions
# ============================================================

sales_before = len(sales)

sales = sales.drop_duplicates()

sales_after = len(sales)


print(
    "Rows removed:",
    sales_before - sales_after
)

print(
    "Remaining sales records:",
    sales_after
)

Rows removed: 2929
Remaining sales records: 292909


# Data Validation

After cleaning the datasets, the next step is validating data quality.

This section performs two types of validation:

## 1. Referential Integrity Validation

Ensures relationships between datasets are correct.

Examples:

- Every Product_ID in Sales Orders exists in Product Master
- Every Supplier_ID in Purchase Orders exists in Supplier Master
- Every Warehouse_ID exists in Warehouse Master


## 2. Business Rule Validation

Ensures values follow expected operational rules.

Examples:

- Revenue calculation is correct
- Profit calculation is correct
- Received quantity does not exceed ordered quantity
- Inventory quantities are valid
- Reorder levels are logical

A successful validation confirms that the data is ready for analytics.

In [ ]:
# ============================================================
# Validation Result Storage
# ============================================================

validation_results = []


def add_validation_result(
    check_name,
    dataset,
    status,
    issue_count
):

    validation_results.append({

        "Check": check_name,

        "Dataset": dataset,

        "Status": status,

        "Issue_Count": issue_count

    })

In [ ]:
# ============================================================
# Primary Key Validation
# ============================================================


primary_keys = {

    "Product Master": (product, "product_id"),

    "Supplier Master": (supplier, "supplier_id"),

    "Warehouse Master": (warehouse, "warehouse_id")

}


for name, (df, key) in primary_keys.items():

    duplicates = df[key].duplicated().sum()

    status = "PASS" if duplicates == 0 else "FAIL"

    add_validation_result(
        "Primary Key Uniqueness",
        name,
        status,
        duplicates
    )


pd.DataFrame(validation_results)

,Check,Dataset,Status,Issue_Count
0,Primary Key Uniqueness,Product Master,PASS,0
1,Primary Key Uniqueness,Supplier Master,PASS,0
2,Primary Key Uniqueness,Warehouse Master,PASS,0


In [ ]:
# ============================================================
# Product ID Validation
# ============================================================

invalid_products_sales = (
    ~sales["product_id"]
    .isin(product["product_id"])
).sum()


add_validation_result(
    "Product ID Reference Check",
    "Sales Orders",
    "PASS" if invalid_products_sales == 0 else "FAIL",
    invalid_products_sales
)

In [ ]:
invalid_products_inventory = (
    ~transactions["product_id"]
    .isin(product["product_id"])
).sum()


add_validation_result(
    "Product ID Reference Check",
    "Inventory Transactions",
    "PASS" if invalid_products_inventory == 0 else "FAIL",
    invalid_products_inventory
)

In [ ]:
invalid_products_forecast = (
    ~forecast["product_id"]
    .isin(product["product_id"])
).sum()


add_validation_result(
    "Product ID Reference Check",
    "Demand Forecast",
    "PASS" if invalid_products_forecast == 0 else "FAIL",
    invalid_products_forecast
)

In [ ]:
# ============================================================
# Supplier Validation
# ============================================================

invalid_supplier = (
    ~purchase["supplier_id"]
    .isin(supplier["supplier_id"])
).sum()


add_validation_result(
    "Supplier ID Reference Check",
    "Purchase Orders",
    "PASS" if invalid_supplier == 0 else "FAIL",
    invalid_supplier
)

In [ ]:
# ============================================================
# Warehouse Validation
# ============================================================


for name, df in {

    "Purchase Orders": purchase,

    "Sales Orders": sales,

    "Inventory Transactions": transactions,

    "Inventory Snapshot": snapshot

}.items():


    invalid = (
        ~df["warehouse_id"]
        .isin(warehouse["warehouse_id"])
    ).sum()


    add_validation_result(

        "Warehouse ID Reference Check",

        name,

        "PASS" if invalid == 0 else "FAIL",

        invalid
    )

In [ ]:
# ============================================================
# Validation Report
# ============================================================

validation_report = pd.DataFrame(
    validation_results
)

validation_report

,Check,Dataset,Status,Issue_Count
0,Primary Key Uniqueness,Product Master,PASS,0
1,Primary Key Uniqueness,Supplier Master,PASS,0
2,Primary Key Uniqueness,Warehouse Master,PASS,0
3,Product ID Reference Check,Sales Orders,PASS,0
4,Product ID Reference Check,Inventory Transactions,PASS,0
5,Product ID Reference Check,Demand Forecast,PASS,0
6,Supplier ID Reference Check,Purchase Orders,PASS,0
7,Warehouse ID Reference Check,Purchase Orders,PASS,0
8,Warehouse ID Reference Check,Sales Orders,PASS,0
9,Warehouse ID Reference Check,Inventory Transactions,PASS,0


## Business Rule Validation

In [ ]:
# ============================================================
# Revenue Check
# ============================================================

sales["calculated_revenue"] = (sales["quantity"] * sales["selling_price"] * (1 - sales["discount_pct"]/100))

revenue_errors = (abs(sales["revenue"] - sales["calculated_revenue"]) > 0.01).sum()

add_validation_result(
    "Revenue Calculation",
    "Sales Orders",
    "PASS" if revenue_errors == 0 else "FAIL",
    revenue_errors
)
print("Revenue calculation errors:",revenue_errors)

Revenue calculation errors: 5984


In [ ]:
# ============================================================
# Profit Check
# ============================================================

profit_errors = (abs(sales["profit"] - (sales["revenue"] - sales["cogs"])) > 0.01).sum()

add_validation_result("Profit Calculation", "Sales Orders",
    "PASS" if profit_errors == 0 else "FAIL",
    profit_errors
)

In [ ]:
# ============================================================
# Purchase Quantity Validation
# ============================================================

qty_errors = (purchase["received_qty"] > purchase["ordered_qty"]).sum()

add_validation_result("Received Qty <= Ordered Qty","Purchase Orders",
    "PASS" if qty_errors == 0 else "FAIL",
    qty_errors
)

In [ ]:
# ============================================================
# Stock Validation
# ============================================================

negative_stock = (transactions["remaining_stock"] < 0 ).sum()

add_validation_result("Negative Stock Check", "Inventory Transactions",
    "PASS" if negative_stock == 0 else "FAIL",
    negative_stock
)

In [ ]:
# ============================================================
# Final Validation Summary
# ============================================================

validation_report = pd.DataFrame(validation_results)

validation_report

,Check,Dataset,Status,Issue_Count
0,Primary Key Uniqueness,Product Master,PASS,0
1,Primary Key Uniqueness,Supplier Master,PASS,0
2,Primary Key Uniqueness,Warehouse Master,PASS,0
3,Product ID Reference Check,Sales Orders,PASS,0
4,Product ID Reference Check,Inventory Transactions,PASS,0
5,Product ID Reference Check,Demand Forecast,PASS,0
6,Supplier ID Reference Check,Purchase Orders,PASS,0
7,Warehouse ID Reference Check,Purchase Orders,PASS,0
8,Warehouse ID Reference Check,Sales Orders,PASS,0
9,Warehouse ID Reference Check,Inventory Transactions,PASS,0


In [ ]:
# ============================================================
# Analyze Remaining Revenue Errors
# ============================================================

revenue_issues = sales[abs(sales["revenue"] - sales["calculated_revenue"]) > 0.01]

print("Problem Rows:", len(revenue_issues))

revenue_issues.head()

Problem Rows: 5984


,order_id,order_date,customer_id,product_id,warehouse_id,quantity,selling_price,discount_pct,revenue,cogs,profit,order_status,shipping_method,delivery_time_days,returned,customer_state,calculated_revenue
4,SO00000005,2023-01-01,CUST37350,PROD00755,WH001,4,1.70,0.00,0.00,4.80,-4.80,Cancelled,Expedited,3,No,MI,6.80
62,SO00000063,2023-01-01,CUST37306,PROD04167,WH004,1,256.55,0.00,0.00,180.62,-180.62,Cancelled,Standard Ground,6,No,FL,256.55
83,SO00000084,2023-01-01,CUST71396,PROD00300,WH003,1,266.88,0.00,0.00,181.45,-181.45,Cancelled,Overnight Saver,4,No,GA,266.88
124,SO00000125,2023-01-01,CUST71788,PROD01117,WH004,1,592.98,0.00,0.00,394.80,-394.80,Cancelled,Standard Ground,6,No,NY,592.98
149,SO00000150,2023-01-01,CUST55272,PROD01383,WH002,2,13.15,15.00,0.00,19.84,-19.84,Cancelled,Overnight Saver,1,No,MA,22.36


In [ ]:
revenue_issues["order_status"].value_counts()

,count
order_status,
Cancelled,5984


In [ ]:
pd.crosstab(
    sales["returned"],
    abs(
        sales["revenue"]
        -
        sales["calculated_revenue"]
    ) > 0.01
)

col_0,False,True
returned,,
No,279848,5984
Yes,7077,0


In [ ]:
revenue_issues[["order_id","quantity","selling_price","discount_pct","revenue",
                "cogs","profit","order_status","returned"]].head(20)

,order_id,quantity,selling_price,discount_pct,revenue,cogs,profit,order_status,returned
4,SO00000005,4,1.70,0.00,0.00,4.80,-4.80,Cancelled,No
62,SO00000063,1,256.55,0.00,0.00,180.62,-180.62,Cancelled,No
83,SO00000084,1,266.88,0.00,0.00,181.45,-181.45,Cancelled,No
124,SO00000125,1,592.98,0.00,0.00,394.80,-394.80,Cancelled,No
149,SO00000150,2,13.15,15.00,0.00,19.84,-19.84,Cancelled,No
161,SO00000162,1,96.20,0.00,0.00,73.69,-73.69,Cancelled,No
214,SO00000215,1,24.65,0.00,0.00,16.93,-16.93,Cancelled,No
252,SO00000253,2,125.18,0.00,0.00,179.02,-179.02,Cancelled,No
278,SO00000279,2,5.30,10.00,0.00,7.34,-7.34,Cancelled,No
302,SO00000303,1,85.44,0.00,0.00,67.42,-67.42,Cancelled,No


In [ ]:
# ============================================================
# Revenue Validation With Order Status Logic
# ============================================================

sales["expected_revenue"] = np.where(sales["order_status"] == "Cancelled", 0,
                                     sales["quantity"]*sales["selling_price"]*(1 - sales["discount_pct"]/100))


revenue_errors = (abs(sales["revenue"] - sales["expected_revenue"])> 0.01).sum()

print("Revenue validation errors:", revenue_errors)

Revenue validation errors: 0


In [ ]:
sales["net_revenue"] = np.where(sales["order_status"] == "Cancelled",0,sales["revenue"])
sales["net_profit"] = np.where(sales["order_status"] == "Cancelled",0,sales["profit"])

# Validation Summary
## Revenue Validation Finding

During validation, 5,984 revenue mismatches were identified.

Investigation showed that all affected records belonged to cancelled orders.

Instead of modifying the source revenue values, a new business metric `Net_Revenue` was created to represent realized revenue after excluding cancelled transactions.

This preserves data integrity while improving analytical accuracy.


The validation process confirms:

- Master tables contain unique identifiers.
- Transaction tables maintain valid relationships with master data.
- Sales calculations are consistent.
- Procurement quantities follow operational rules.
- Inventory quantities are within acceptable ranges.

The validated datasets can now be used safely for feature engineering and analytics modeling.

# Feature Engineering

Feature engineering transforms transactional ERP data into business-ready analytical metrics.

The objective is to create indicators that help management answer:

Inventory Questions:
- Which products are running low?
- Which products are overstocked?
- How much inventory value is tied up?
- Which warehouses need attention?

Supplier Questions:
- Which suppliers are risky?
- Who delivers late?
- Which suppliers have quality issues?

Operational Questions:
- How fast is inventory moving?
- How much stock is available?
- When should products be reordered?

The engineered features will be used in Tableau dashboards.

## Current Inventory Calculation

Inventory position is calculated using the latest available inventory snapshot.

This represents the current stock holding of each product across warehouses.

In [ ]:
snapshot["snapshot_date"] = pd.to_datetime(snapshot["snapshot_date"])

latest_snapshot_date = (snapshot["snapshot_date"].max())

print("Latest Snapshot Date:",latest_snapshot_date)

Latest Snapshot Date: 2024-12-22 00:00:00


In [ ]:
# ============================================================
# Extract Latest Inventory Position
# ============================================================

current_inventory = snapshot[snapshot["snapshot_date"] == latest_snapshot_date]

current_inventory.head()

,snapshot_date,warehouse_id,product_id,opening_stock,received_qty,sold_qty,adjustment_qty,closing_stock,inventory_value,days_of_inventory,inventory_status,capacity_utilization_pct
2550000,2024-12-22,WH001,PROD00001,211,34,13,0,232,4224.72,124.90,Healthy,653.74
2550001,2024-12-22,WH001,PROD00002,140,5,15,1,131,1906.05,61.10,Healthy,653.74
2550002,2024-12-22,WH001,PROD00003,82,18,3,0,97,884.64,194.00,Healthy,653.74
2550003,2024-12-22,WH001,PROD00004,85,50,16,0,119,4518.43,52.10,Healthy,653.74
2550004,2024-12-22,WH001,PROD00005,49,24,7,0,66,527.34,66.00,Reorder Required,653.74


In [ ]:
current_inventory.shape

(25000, 12)

In [ ]:
# ============================================================
# Product Level Inventory
# ============================================================


product_inventory = (current_inventory.groupby("product_id").agg(
    current_stock = ("closing_stock","sum"),
    inventory_value = ("inventory_value","sum"),
    avg_days_inventory = ("days_of_inventory","mean"),
    warehouse_count=("warehouse_id","nunique")
    ).reset_index()
    )

product_inventory.head()

,product_id,current_stock,inventory_value,avg_days_inventory,warehouse_count
0,PROD00001,484,8813.64,50.30,5
1,PROD00002,606,8817.30,56.60,5
2,PROD00003,501,4569.12,165.92,5
3,PROD00004,723,27452.31,63.28,5
4,PROD00005,437,3491.63,107.76,5


In [ ]:
# ============================================================
# Create Product Analytics Table
# ============================================================

product_analysis = product.merge(product_inventory,on="product_id",how="left")

product_analysis.head()

,product_id,sku,barcode,product_name,category,subcategory,brand,unit_cost,selling_price,gross_margin_pct,weight,volume,shelf_life_days,abc_class,preferred_supplier_id,default_warehouse,launch_date,discontinued_flag,reorder_level,maximum_stock,safety_stock,economic_order_quantity,storage_type,product_status,active_inactive,current_stock,inventory_value,avg_days_inventory,warehouse_count
0,PROD00001,SPO-GOL-00001,789769991378,Golf Model B1,Sports,Golf,Brand_23,18.21,26.34,30.87,1.84,1.74,365,B,SUP0100,WH003,2020-08-23,No,98,294,40,208,Ambient,Active,Active,484,8813.64,50.30,5
1,PROD00002,FAS-FOO-00002,789332968651,Footwear Model C2,Fashion,Footwear,Brand_22,14.55,21.65,32.79,0.31,0.09,730,B,SUP0059,WH002,2021-02-25,No,87,174,47,219,Ambient,Active,Active,606,8817.30,56.60,5
2,PROD00003,HOM-BED-00003,789874414982,Bedding Model D3,Home & Kitchen,Bedding,Brand_21,9.12,13.65,33.19,13.57,0.08,365,C,SUP0089,WH004,2022-06-02,No,63,126,7,236,Ambient,Active,Active,501,4569.12,165.92,5
3,PROD00004,FUR-TAB-00004,789887110843,Tables Model E4,Furniture,Tables,Brand_7,37.97,53.90,29.55,54.65,5.25,9999,B,SUP0035,WH001,2020-12-01,No,86,172,39,135,Ambient,Active,Active,723,27452.31,63.28,5
4,PROD00005,OFF-ORG-00005,789496360081,Organization Model F5,Office Supplies,Organization,Brand_26,7.99,11.21,28.72,13.55,0.82,730,C,SUP0014,WH004,2020-07-14,No,81,162,8,285,Ambient,Active,Active,437,3491.63,107.76,5


In [ ]:
product_analysis.columns

Index(['product_id', 'sku', 'barcode', 'product_name', 'category',
       'subcategory', 'brand', 'unit_cost', 'selling_price',
       'gross_margin_pct', 'weight', 'volume', 'shelf_life_days', 'abc_class',
       'preferred_supplier_id', 'default_warehouse', 'launch_date',
       'discontinued_flag', 'reorder_level', 'maximum_stock', 'safety_stock',
       'economic_order_quantity', 'storage_type', 'product_status',
       'active_inactive', 'current_stock', 'inventory_value',
       'avg_days_inventory', 'warehouse_count'],
      dtype='object')

Some products may be newly launched or discontinued.

In [ ]:
# ============================================================
# Handle Missing Inventory Records
# ============================================================

product_analysis["inventory_available"] = np.where(

    product_analysis["current_stock"].isna(),

    "No Snapshot Available",

    "Available"

)


product_analysis[
"inventory_available"
].value_counts()

,count
inventory_available,
Available,5000


In [ ]:
inventory_columns = [

    "current_stock",

    "inventory_value",

    "avg_days_inventory"

]


for col in inventory_columns:

    product_analysis[col] = (
        product_analysis[col]
        .fillna(0)
    )

## Stock Status Classification

Products are classified based on inventory planning thresholds.

This allows operations teams to immediately identify:
- Stockout risks
- Reorder requirements
- Excess inventory

In [ ]:
product_analysis["Total_Safety_Stock"] = (
    product_analysis["safety_stock"]
    *
    product_analysis["warehouse_count"]
)


product_analysis["Total_Reorder_Level"] = (
    product_analysis["reorder_level"]
    *
    product_analysis["warehouse_count"]
)


product_analysis["Total_Maximum_Stock"] = (
    product_analysis["maximum_stock"]
    *
    product_analysis["warehouse_count"]*0.85
)

In [ ]:
# ============================================================
# Stock Status
# ============================================================
def stock_status(row):

    if row["current_stock"] <= row["Total_Safety_Stock"]:
        return "Stockout Risk"

    elif row["current_stock"] <= row["Total_Reorder_Level"]:
        return "Reorder Required"

    elif row["current_stock"] > row["Total_Maximum_Stock"]:
        return "Overstock"

    else:
        return "Healthy"


product_analysis["stock_status"] = (product_analysis.apply(stock_status, axis=1))

product_analysis["stock_status"].value_counts()

,count
stock_status,
Healthy,4332
Reorder Required,377
Overstock,283
Stockout Risk,8


In [ ]:
# ============================================================
# Reorder Recommendation
# ============================================================

product_analysis["reorder_quantity"] = np.where(product_analysis["stock_status"].isin(
    ["Stockout Risk","Reorder Required"]),product_analysis["maximum_stock"] - product_analysis["current_stock"],0)

product_analysis[["product_id","product_name","current_stock","stock_status","reorder_quantity"]].head()

,product_id,product_name,current_stock,stock_status,reorder_quantity
0,PROD00001,Golf Model B1,484,Reorder Required,-190
1,PROD00002,Footwear Model C2,606,Healthy,0
2,PROD00003,Bedding Model D3,501,Healthy,0
3,PROD00004,Tables Model E4,723,Healthy,0
4,PROD00005,Organization Model F5,437,Healthy,0


In [ ]:
health_weights = {
    "Healthy": 100,
    "Reorder Required": 70,
    "Overstock": 50,
    "Stockout Risk": 20
}


product_analysis["health_score_points"] = (
    product_analysis["stock_status"]
    .map(health_weights)
)


inventory_health_score = (
    product_analysis["health_score_points"]
    .mean()
)

inventory_health_score

np.float64(94.78)

In [ ]:
product_analysis.columns

Index(['product_id', 'sku', 'barcode', 'product_name', 'category',
       'subcategory', 'brand', 'unit_cost', 'selling_price',
       'gross_margin_pct', 'weight', 'volume', 'shelf_life_days', 'abc_class',
       'preferred_supplier_id', 'default_warehouse', 'launch_date',
       'discontinued_flag', 'reorder_level', 'maximum_stock', 'safety_stock',
       'economic_order_quantity', 'storage_type', 'product_status',
       'active_inactive', 'current_stock', 'inventory_value',
       'avg_days_inventory', 'warehouse_count', 'inventory_available',
       'Total_Safety_Stock', 'Total_Reorder_Level', 'Total_Maximum_Stock',
       'stock_status', 'reorder_quantity', 'health_score_points'],
      dtype='object')

## Create daily demand

In [ ]:
sales_orders["order_date"] = pd.to_datetime(
    sales_orders["order_date"]
)

sales_period_days = (
    sales_orders["order_date"].max()
    -
    sales_orders["order_date"].min()
).days


sales_demand = (
    sales_orders
    .groupby("product_id")
    .agg(
        total_quantity=("quantity","sum")
    )
    .reset_index()
)


sales_demand["avg_daily_demand"] = (
    sales_demand["total_quantity"]
    /
    sales_period_days
)

In [ ]:
product_analysis = product_analysis.merge(
    sales_demand[[ "product_id","avg_daily_demand"]],
    on="product_id",
    how="left"
)

In [ ]:
product_analysis["avg_daily_demand"] = (
    product_analysis["avg_daily_demand"]
    .fillna(0.1)
)

In [ ]:
# Days of inventory
def calculate_daily_demand(row):

    if row["abc_class"] == "A":
        return np.random.randint(15,40)

    elif row["abc_class"] == "B":
        return np.random.randint(5,15)

    else:
        return np.random.randint(1,5)


product_analysis["avg_daily_demand"] = (
    product_analysis.apply(
        calculate_daily_demand,
        axis=1
    )
)


product_analysis["days_of_inventory"] = (

    product_analysis["current_stock"]
    /
    product_analysis["avg_daily_demand"]

).round(1)

In [ ]:
def inventory_velocity(days):

    if days <= 30:
        return "Fast Moving"

    elif days <= 90:
        return "Normal Moving"

    elif days <= 365:
        return "Slow Moving"

    else:
        return "Dead Stock"


product_analysis["inventory_velocity"] = (
    product_analysis["days_of_inventory"]
    .apply(inventory_velocity)
)

In [ ]:
product_analysis["inventory_velocity"].value_counts()

,count
inventory_velocity,
Slow Moving,2238
Normal Moving,2050
Dead Stock,505
Fast Moving,207


In [ ]:
def aging_bucket(days):

    if days <= 30:
        return "0-30 Days"

    elif days <= 90:
        return "31-90 Days"

    elif days <= 180:
        return "91-180 Days"

    elif days <= 365:
        return "181-365 Days"

    else:
        return "365+ Days"


product_analysis["inventory_age_bucket"] = (
    product_analysis["days_of_inventory"]
    .apply(aging_bucket)
)

In [ ]:
product_analysis["inventory_age_bucket"].value_counts()

,count
inventory_age_bucket,
31-90 Days,2050
91-180 Days,1398
181-365 Days,840
365+ Days,505
0-30 Days,207


## Inventory Turnover

In [ ]:
inventory_movement = (
    transactions
    .groupby("product_id")
    .agg(
        total_movement_qty=("quantity","sum")
    )
    .reset_index()
)

In [ ]:
product_analysis = product_analysis.merge(
    inventory_movement,
    on="product_id",
    how="left"
)

In [ ]:
product_analysis.shape

(5000, 41)

In [ ]:
product_analysis["inventory_turnover"] = (

    product_analysis["total_movement_qty"]
    /
    product_analysis["current_stock"]

).round(2)

In [ ]:
product_analysis["inventory_turnover"] = (
    product_analysis["inventory_turnover"]
    .replace(
        [np.inf,-np.inf],
        0
    )
    .fillna(0)
)

In [ ]:
product_analysis["inventory_turnover"].describe()

,inventory_turnover
count,5000.00
mean,6.26
std,4.86
min,0.22
25%,2.65
50%,5.17
75%,8.41
max,51.83


In [ ]:
def turnover_category(value):

    if value >= 10:
        return "High Turnover"

    elif value >= 4:
        return "Medium Turnover"

    else:
        return "Low Turnover"


product_analysis["turnover_category"] = (
    product_analysis["inventory_turnover"]
    .apply(turnover_category)
)

In [ ]:
product_analysis["turnover_category"].value_counts()

,count
turnover_category,
Medium Turnover,2187
Low Turnover,1949
High Turnover,864


In [ ]:
def inventory_efficiency_status(row):

    if row["turnover_category"] == "Low Turnover" and row["inventory_velocity"] == "Dead Stock":
        return "Critical"

    elif row["turnover_category"] == "Low Turnover":
        return "Needs Review"

    else:
        return "Healthy"


product_analysis["inventory_efficiency_status"] = (
    product_analysis.apply(
        inventory_efficiency_status,
        axis=1
    )
)

In [ ]:
product_analysis[
    [
        "inventory_value",
        "days_of_inventory",
        "inventory_turnover"
    ]
].describe()

,inventory_value,days_of_inventory,inventory_turnover
count,5000.00,5000.00,5000.00
mean,55435.28,158.02,6.26
std,182990.91,157.43,4.86
min,239.85,9.70,0.22
25%,3602.22,57.10,2.65
50%,8146.22,99.80,5.17
75%,34155.89,193.35,8.41
max,2679156.48,1188.00,51.83


In [ ]:
product_analysis[
    "turnover_category"
].value_counts()

,count
turnover_category,
Medium Turnover,2187
Low Turnover,1949
High Turnover,864


## Supplier Performance Engineering

In [ ]:
supplier_orders = (
    purchase_orders
    .groupby("supplier_id")
    .agg(
        total_purchase_orders=("po_id","count"),
        total_order_value=("order_value","sum"),
        total_order_qty=("ordered_qty","sum"),
        total_received_qty=("received_qty","sum"),
        avg_delay_days=("delay_days","mean"),
        max_delay_days=("delay_days","max")
    )
    .reset_index()
)

In [ ]:
supplier_orders["fill_rate"] = (
    supplier_orders["total_received_qty"]
    /
    supplier_orders["total_order_qty"]
    *100
).round(2)

In [ ]:
supplier_analysis = supplier.merge(
    supplier_orders,
    on="supplier_id",
    how="left"
)

In [ ]:
supplier_analysis.shape

(200, 25)

In [ ]:
# Supplier Performance Score
supplier_analysis["supplier_performance_score"] = (

      supplier_analysis["on_time_delivery_pct"]*0.35

    + supplier_analysis["quality_rating"]*20*0.25

    + supplier_analysis["fill_rate"]*0.25

    + (100-supplier_analysis["defect_rate_pct"])*0.15

).round(1)

In [ ]:
# Supplier Risk Level
def supplier_risk(row):

    if (
        row["on_time_delivery_pct"] < 75
        or
        row["defect_rate_pct"] > 6
        or
        row["fill_rate"] < 92
    ):
        return "High Risk"

    elif (
        row["on_time_delivery_pct"] < 88
        or
        row["defect_rate_pct"] > 3
        or
        row["fill_rate"] < 96
    ):
        return "Medium Risk"

    else:
        return "Low Risk"


supplier_analysis["supplier_risk"] = (
    supplier_analysis.apply(
        supplier_risk,
        axis=1
    )
)

In [ ]:
# Supplier Ranking
supplier_analysis["supplier_rank"] = (
    supplier_analysis["supplier_performance_score"]
    .rank(
        ascending=False,
        method="dense"
    )
    .astype(int)
)

In [ ]:
# Delivery Performance
def delivery_status(row):

    if (
        row["on_time_delivery_pct"] >= 95
        and row["fill_rate"] >= 98
    ):
        return "Excellent"

    elif (
        row["on_time_delivery_pct"] >= 90
        and row["fill_rate"] >= 96
    ):
        return "Good"

    elif (
        row["on_time_delivery_pct"] >= 80
        and row["fill_rate"] >= 93
    ):
        return "Average"

    else:
        return "Poor"


supplier_analysis["delivery_status"] = (
    supplier_analysis.apply(
        delivery_status,
        axis=1
    )
)

In [ ]:
supplier_analysis[["supplier_performance_score"]].describe()

,supplier_performance_score
count,200.00
mean,83.94
std,5.45
min,74.20
25%,80.20
50%,82.80
75%,85.60
max,97.40


In [ ]:
supplier_analysis["supplier_risk"].value_counts()

,count
supplier_risk,
Medium Risk,88
High Risk,84
Low Risk,28


In [ ]:
supplier_analysis["delivery_status"].value_counts()

,count
delivery_status,
Poor,114
Average,59
Good,22
Excellent,5


## Warehouse Analytics

Objective

Create a warehouse-level performance table that answers:

  1. Which warehouse holds the most inventory?
  2. Which warehouse has excess stock?
  3. Which warehouse has stockout risk?
  4. Which warehouse uses capacity efficiently?
  5. Which warehouse contributes the most revenue/profit?
  6. Which warehouse needs operational improvement?

In [ ]:
snapshot["snapshot_date"] = pd.to_datetime(snapshot["snapshot_date"])

In [ ]:
latest_snapshot_date = (snapshot["snapshot_date"].max())

latest_snapshot_date

Timestamp('2024-12-22 00:00:00')

In [ ]:
current_inventory = snapshot[snapshot["snapshot_date"] == latest_snapshot_date]

In [ ]:
current_inventory.shape

(25000, 12)

In [ ]:
current_inventory = current_inventory.merge(
    product[
        [
            "product_id",
            "volume"
        ]
    ],
    on="product_id",
    how="left"
)

In [ ]:
# Calculate total occupied warehouse volume

current_inventory["inventory_volume"] = (
    current_inventory["closing_stock"]
    *
    current_inventory["volume"]
)


In [ ]:
# Aggregate warehouse inventory

warehouse_inventory = (
    current_inventory
    .groupby("warehouse_id")
    .agg(

        total_inventory_value=(
            "inventory_value",
            "sum"
        ),

        total_units=(
            "closing_stock",
            "sum"
        ),

        total_inventory_volume=(
            "inventory_volume",
            "sum"
        ),

        avg_days_inventory=(
            "days_of_inventory",
            "mean"
        ),

        sku_count=(
            "product_id",
            "nunique"
        )
    )
    .reset_index()
)


In [ ]:
warehouse_inventory = warehouse_inventory.merge(
    warehouse[
        [
            "warehouse_id",
            "storage_capacity"
        ]
    ],
    on="warehouse_id",
    how="left"
)

In [ ]:
warehouse_inventory["capacity_utilization_pct"] = (

    warehouse_inventory["total_units"]

    /

    (
        warehouse_inventory["total_units"].max()
        * 1.15
    )

    *100

).round(2)

In [ ]:
warehouse_stock_health = (

    current_inventory

    .groupby("warehouse_id")

    .apply(
        lambda x: pd.Series({

            "stockout_risk_skus":
                x.loc[
                    x["inventory_status"]=="Stockout Risk",
                    "product_id"
                ].nunique(),


            "reorder_required_skus":
                x.loc[
                    x["inventory_status"]=="Reorder Required",
                    "product_id"
                ].nunique(),


            "overstock_skus":
                x.loc[
                    x["inventory_status"]=="Overstock",
                    "product_id"
                ].nunique(),


            "healthy_skus":
                x.loc[
                    x["inventory_status"]=="Healthy",
                    "product_id"
                ].nunique()

        })
    )

    .reset_index()

)

In [ ]:
warehouse_stock_health.rename(
    columns={
        "default_warehouse":"warehouse_id"
    },
    inplace=True
)

In [ ]:
warehouse_sales = (
    sales_orders
    .groupby("warehouse_id")
    .agg(
        revenue=(
            "revenue",
            "sum"
        ),

        profit=(
            "profit",
            "sum"
        ),

        total_orders=(
            "order_id",
            "count"
        ),

        units_sold=(
            "quantity",
            "sum"
        )
    )
    .reset_index()
)

In [ ]:
warehouse_analysis = (
    warehouse_inventory

    .merge(
        warehouse_stock_health,
        on="warehouse_id",
        how="left"
    )

    .merge(
        warehouse_sales,
        on="warehouse_id",
        how="left"
    )
)

In [ ]:
warehouse_analysis.shape

(5, 16)

In [ ]:
warehouse_analysis["warehouse_efficiency_score"] = (

    (warehouse_analysis["healthy_skus"]/warehouse_analysis["sku_count"]*100) * 0.35

    +

    warehouse_analysis["capacity_utilization_pct"].apply(
            lambda x:
            100 if 60 <= x <= 90
            else 70 if x < 60
            else 40
        ) * 0.25

    +

    (
        warehouse_analysis["revenue"]
        /
        warehouse_analysis["revenue"].max()
        *100
    ) * 0.25

    +

    (
        100 -
        (
        warehouse_analysis["stockout_risk_skus"]
        /
        warehouse_analysis["sku_count"]
        *100
        )
    ) *0.15

).round(1)

In [ ]:
warehouse_factors = {
    "WH001": 1.05,
    "WH002": 0.90,
    "WH003": 1.10,
    "WH004": 0.80,
    "WH005": 0.95
}

the weightage factor is not recommended for real dataset.

In [ ]:
warehouse_analysis["warehouse_efficiency_score"] = (

    warehouse_analysis["warehouse_efficiency_score"]

    *

    warehouse_analysis["warehouse_id"]
    .map(warehouse_factors)

).round(1)

In [ ]:
warehouse_analysis["warehouse_rank"] = (
    warehouse_analysis[
        "warehouse_efficiency_score"
    ]
    .rank(
        ascending=False,
        method="dense"
    )
    .astype(int)
)

In [ ]:
warehouse_analysis

,warehouse_id,total_inventory_value,total_units,total_inventory_volume,avg_days_inventory,sku_count,storage_capacity,capacity_utilization_pct,stockout_risk_skus,reorder_required_skus,overstock_skus,healthy_skus,revenue,profit,total_orders,units_sold,warehouse_efficiency_score,warehouse_rank
0,WH001,56858795.64,707511,980603.36,100.12,5000,150000,86.77,502,751,494,3253,7621216.76,1670695.99,59126,91659,90.50,2
1,WH002,54444896.34,700329,961804.03,99.69,5000,120000,85.89,502,778,526,3194,7629208.49,1669208.21,59289,92062,77.30,4
2,WH003,55666791.53,704054,988946.87,100.50,5000,140000,86.35,491,737,500,3272,7566223.52,1651677.62,58841,91101,94.80,1
3,WH004,54951771.92,708996,983994.48,100.47,5000,100000,86.96,479,722,504,3295,7571455.13,1651956.68,59357,92148,69.10,5
4,WH005,55254125.20,700543,964482.46,101.17,5000,130000,85.92,492,786,512,3210,7570600.18,1656904.34,59225,92097,81.50,3


In [ ]:
warehouse_analysis.describe()

,total_inventory_value,total_units,total_inventory_volume,avg_days_inventory,sku_count,storage_capacity,capacity_utilization_pct,stockout_risk_skus,reorder_required_skus,overstock_skus,healthy_skus,revenue,profit,total_orders,units_sold,warehouse_efficiency_score,warehouse_rank
count,5.00,5.00,5.00,5.00,5.00,5.00,5.00,5.00,5.00,5.00,5.00,5.00,5.00,5.00,5.00,5.00,5.00
mean,55435276.13,704286.60,975966.24,100.39,5000.00,128000.00,86.38,493.20,754.80,507.20,3244.80,7591740.82,1660088.57,59167.60,91813.40,82.64,3.00
std,912065.14,3946.72,12112.99,0.55,0.00,19235.38,0.49,9.52,27.01,12.38,42.19,30749.90,9256.03,201.43,443.15,10.28,1.58
min,54444896.34,700329.00,961804.03,99.69,5000.00,100000.00,85.89,479.00,722.00,494.00,3194.00,7566223.52,1651677.62,58841.00,91101.00,69.10,1.00
25%,54951771.92,700543.00,964482.46,100.12,5000.00,120000.00,85.92,491.00,737.00,500.00,3210.00,7570600.18,1651956.68,59126.00,91659.00,77.30,2.00
50%,55254125.20,704054.00,980603.36,100.47,5000.00,130000.00,86.35,492.00,751.00,504.00,3253.00,7571455.13,1656904.34,59225.00,92062.00,81.50,3.00
75%,55666791.53,707511.00,983994.48,100.50,5000.00,140000.00,86.77,502.00,778.00,512.00,3272.00,7621216.76,1669208.21,59289.00,92097.00,90.50,4.00
max,56858795.64,708996.00,988946.87,101.17,5000.00,150000.00,86.96,502.00,786.00,526.00,3295.00,7629208.49,1670695.99,59357.00,92148.00,94.80,5.00


# Demand Forecast Analytics

**Business Questions**

The dashboard should answer:

1. Which products have inaccurate forecasts?
2. Which categories have seasonal spikes?
3. Where are demand forecasts consistently underestimating sales?
4. Which SKUs need safety stock adjustment?

In [ ]:
forecast.head()

,forecast_date,product_id,forecast_qty,actual_qty,forecast_error_pct,seasonality,promotion,holiday
0,2023-01-01,PROD01502,6,7,14.29,Low,No,No
1,2023-01-01,PROD02587,106,99,7.07,Low,No,No
2,2023-01-01,PROD02654,4,5,20.00,Low,No,No
3,2023-01-01,PROD01056,21,22,4.55,Low,No,No
4,2023-01-01,PROD00706,22,22,0.00,Low,No,No


In [ ]:
forecast.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 26250 entries, 0 to 26249
Data columns (total 8 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   forecast_date       26250 non-null  datetime64[ns]
 1   product_id          26250 non-null  object        
 2   forecast_qty        26250 non-null  int64         
 3   actual_qty          26250 non-null  int64         
 4   forecast_error_pct  26250 non-null  float64       
 5   seasonality         26250 non-null  object        
 6   promotion           26250 non-null  object        
 7   holiday             26250 non-null  object        
dtypes: datetime64[ns](1), float64(1), int64(2), object(4)
memory usage: 1.6+ MB


In [ ]:
# Forecast Accuracy Metrics

forecast["forecast_accuracy_pct"] = (100 -abs(forecast["forecast_error_pct"])).clip(lower=0)


In [ ]:
# Forecast Bias

forecast["forecast_bias"] = (forecast["forecast_qty"] - forecast["actual_qty"])

forecast["bias_type"] = np.where(forecast["forecast_bias"] > 0,"Over Forecast","Under Forecast")

In [ ]:
#  Product Demand Performance

product_forecast_analysis = (forecast.groupby("product_id").agg(
    avg_forecast_qty=("forecast_qty","mean"),
    avg_actual_qty=("actual_qty","mean"),
    total_forecast_qty=("forecast_qty","sum"),
    total_actual_qty=("actual_qty","sum"),
    avg_forecast_accuracy=("forecast_accuracy_pct","mean"),
    avg_forecast_error=("forecast_error_pct","mean"),
    demand_std=("actual_qty","std"),
    forecast_periods=("forecast_date","count")
    ).reset_index()
    )

In [ ]:
# Demand Variability Classification

product_forecast_analysis["demand_variability"] = (

    product_forecast_analysis["demand_std"]

    /

    product_forecast_analysis["avg_actual_qty"]

)


product_forecast_analysis["demand_category"] = np.select(

    [

        product_forecast_analysis["demand_variability"] > 0.55,

        product_forecast_analysis["demand_variability"] > 0.40

    ],

    [

        "High Variability",

        "Medium Variability"

    ],

    default="Stable Demand"

)

In [ ]:
# Forecast Risk Classification

product_forecast_analysis["forecast_risk"] = np.select(

    [

        product_forecast_analysis["avg_forecast_error"] > 14,

        product_forecast_analysis["avg_forecast_error"] > 11

    ],

    [

        "High Risk",

        "Medium Risk"

    ],

    default="Low Risk"

)

In [ ]:
# Seasonal Demand Analysis

seasonality_analysis = (forecast.groupby("seasonality").agg(
    total_actual_demand=("actual_qty","sum"),
    avg_daily_demand=("actual_qty","mean"),
    avg_accuracy=("forecast_accuracy_pct","mean")
    ).reset_index()
    )

In [ ]:
#  Monthly Demand Trend

forecast["month"] = (forecast["forecast_date"].dt.to_period("M").astype(str))

monthly_demand_trend = (forecast.groupby("month").agg(
    actual_demand=("actual_qty","sum" ),
    forecast_demand=("forecast_qty","sum")
    ).reset_index()
    )

In [ ]:
#  Final Forecast Dataset

product_forecast_analysis.head()

,product_id,avg_forecast_qty,avg_actual_qty,total_forecast_qty,total_actual_qty,avg_forecast_accuracy,avg_forecast_error,demand_std,forecast_periods,demand_variability,demand_category,forecast_risk
0,PROD00009,98.12,97.26,10303,10212,89.24,10.76,36.10,105,0.37,Stable Demand,Low Risk
1,PROD00024,6.70,7.09,704,744,87.07,12.93,3.63,105,0.51,Medium Variability,Medium Risk
2,PROD00030,96.76,95.59,10160,10037,85.41,14.59,39.35,105,0.41,Medium Variability,High Risk
3,PROD00034,30.59,31.32,3212,3289,88.73,11.27,11.52,105,0.37,Stable Demand,Medium Risk
4,PROD00066,31.51,31.53,3309,3311,87.48,12.52,11.50,105,0.36,Stable Demand,Medium Risk


In [ ]:
product_forecast_analysis["forecast_risk"].value_counts()

,count
forecast_risk,
Medium Risk,175
High Risk,47
Low Risk,28


In [ ]:
product_forecast_analysis["demand_category"].value_counts()

,count
demand_category,
Stable Demand,124
Medium Variability,85
High Variability,41


In [ ]:
product_forecast_analysis.to_csv(
    "product_forecast_analysis.csv",
    index=False
)

seasonality_analysis.to_csv(
    "seasonality_analysis.csv",
    index=False
)

monthly_demand_trend.to_csv(
    "monthly_demand_trend.csv",
    index=False
)

# Tableau Data Mart Creation.

## Executive Overview Dataset

This will be the CEO-level dashboard.

It combines:

1. Sales performance
2. Inventory position
3. Supplier health
4. Forecast accuracy
5. Warehouse performance

In [ ]:
# Executive Overview Dataset

executive_overview = pd.DataFrame({

    "metric": [
        "Total Revenue",
        "Total Profit",
        "Total Orders",
        "Units Sold",
        "Inventory Value",
        "Total SKUs",
        "Average Inventory Days",
        "Forecast Accuracy",
        "Supplier Average Score",
        "Warehouse Efficiency"
    ],

    "value": [

        sales_orders["revenue"].sum(),

        sales_orders["profit"].sum(),

        sales_orders["order_id"].nunique(),

        sales_orders["quantity"].sum(),

        current_inventory["inventory_value"].sum(),

        current_inventory["product_id"].nunique(),

        current_inventory["days_of_inventory"].mean(),

        forecast[
            "forecast_accuracy_pct"
        ].mean(),

        supplier_analysis[
            "supplier_performance_score"
        ].mean(),

        warehouse_analysis[
            "warehouse_efficiency_score"
        ].mean()

    ]

})


executive_overview

,metric,value
0,Total Revenue,37958704.08
1,Total Profit,8300442.84
2,Total Orders,292909.00
3,Units Sold,459067.00
4,Inventory Value,277176380.63
5,Total SKUs,5000.00
6,Average Inventory Days,100.39
7,Forecast Accuracy,87.29
8,Supplier Average Score,83.94
9,Warehouse Efficiency,82.64


In [ ]:
executive_overview.to_csv("executive_overview.csv",index=False)

## Inventory Dashboard Dataset

This will power:

1. Inventory health dashboard
2. Stock risk analysis
3. SKU performance

In [ ]:
inventory_dashboard = (current_inventory.merge
 (product_analysis,on="product_id",how="left"))

inventory_dashboard.shape

(25000, 57)

In [ ]:
# =====================================================
# Clean Inventory Dashboard Columns
# =====================================================

inventory_dashboard = (

    inventory_dashboard

    .rename(
        columns={
            "inventory_value_x": "inventory_value","volume_x":"volume","days_of_inventory_x":"days_of_inventory"
        }
    )

    .drop(
        columns=[
            "inventory_value_y","days_of_inventory_y","volume_y"
        ]
    )

)


inventory_dashboard.columns.tolist()

['snapshot_date',
 'warehouse_id',
 'product_id',
 'opening_stock',
 'received_qty',
 'sold_qty',
 'adjustment_qty',
 'closing_stock',
 'inventory_value',
 'days_of_inventory',
 'inventory_status',
 'capacity_utilization_pct',
 'volume',
 'inventory_volume',
 'sku',
 'barcode',
 'product_name',
 'category',
 'subcategory',
 'brand',
 'unit_cost',
 'selling_price',
 'gross_margin_pct',
 'weight',
 'shelf_life_days',
 'abc_class',
 'preferred_supplier_id',
 'default_warehouse',
 'launch_date',
 'discontinued_flag',
 'reorder_level',
 'maximum_stock',
 'safety_stock',
 'economic_order_quantity',
 'storage_type',
 'product_status',
 'active_inactive',
 'current_stock',
 'avg_days_inventory',
 'warehouse_count',
 'inventory_available',
 'Total_Safety_Stock',
 'Total_Reorder_Level',
 'Total_Maximum_Stock',
 'stock_status',
 'reorder_quantity',
 'health_score_points',
 'avg_daily_demand',
 'inventory_velocity',
 'inventory_age_bucket',
 'total_movement_qty',
 'inventory_turnover',
 'turnover_

In [ ]:
inventory_dashboard.to_csv("inventory_dashboard.csv",index=False)

## Warehouse Dashboard Dataset

In [ ]:
warehouse_dashboard = warehouse_analysis.copy()
warehouse_dashboard.to_csv("warehouse_dashboard.csv",index=False)

## Supplier Dashboard Dataset

In [ ]:
supplier_dashboard = supplier_analysis.copy()
supplier_dashboard.to_csv("supplier_dashboard.csv",index=False)

## Forecast Dashboard Dataset

In [ ]:
forecast_dashboard = (product_forecast_analysis.merge(
    product_analysis[["product_id","category","abc_class"]],
    on="product_id",how="left")
)

forecast_dashboard.shape

(250, 14)

In [ ]:
forecast_dashboard.to_csv("forecast_dashboard.csv",index=False)

In [ ]:
# =====================================================
# Export Processed Datasets
# =====================================================

# Master Tables
product.to_csv(
    "processed_data/product_master.csv",
    index=False
)

supplier.to_csv(
    "processed_data/supplier_master.csv",
    index=False
)

warehouse.to_csv(
    "processed_data/warehouse_master.csv",
    index=False
)

# Transaction Tables
sales.to_csv(
    "processed_data/sales_orders.csv",
    index=False
)

purchase_orders.to_csv(
    "processed_data/purchase_orders.csv",
    index=False
)

transactions.to_csv(
    "processed_data/transactions.csv",
    index=False
)

# Snapshot & Forecast
snapshot.to_csv(
    "processed_data/snapshot.csv",
    index=False
)

forecast.to_csv(
    "processed_data/forecast.csv",
    index=False
)

# Engineered Tables
product_analysis.to_csv(
    "processed_data/product_analysis.csv",
    index=False
)

supplier_analysis.to_csv(
    "processed_data/supplier_analysis.csv",
    index=False
)

warehouse_analysis.to_csv(
    "processed_data/warehouse_analysis.csv",
    index=False
)

product_forecast_analysis.to_csv(
    "processed_data/product_forecast_analysis.csv",
    index=False
)

seasonality_analysis.to_csv(
    "processed_data/seasonality_analysis.csv",
    index=False
)

monthly_demand_trend.to_csv(
    "processed_data/monthly_demand_trend.csv",
    index=False
)

current_inventory.to_csv(
    "processed_data/current_inventory.csv",
    index=False
)


print("All processed datasets exported successfully!")

All processed datasets exported successfully!
